In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import os
import json
import pickle

In [2]:
# Load Yijun's 5-fold indices
# THIS IS SFOLD NOT KFOLD (variables were historically named)
kfold_pt = '//home/yaoyi/shared/aksdb-covars/aksdb-point-data/experimental_data/sfold5_split_train_test_indices.json'
with open(kfold_pt, 'r') as file:
    kfold_indices = json.load(file)

KFOLD INDICES STRUCTURE:
Dict
- first level are keys labeled fold_0 to fold_1
    - in each fold_i value is another dictionary where the keys are 'train' and 'test'
        - the value is a list of indices

In [7]:
# Load in the geojson with data
pt_data_pt = '/home/yaoyi/chen7924/peatland_permafrost/data/point_pixel_satbands_gt_epsg4326_v3_removednodata.geojson'
point_data_gdf = gpd.read_file(pt_data_pt)
print(point_data_gdf.columns)

# Load in gt with corresponding tax points
json_link_pt = '/home/yaoyi/shared/aksdb-covars/aksdb-point-data/mapped-aksdb-point-data-v0.1.json'
with open(json_link_pt, 'r') as file:
    gt_data = json.load(file)

Index(['id', 'aksdb_dts', 'x_3338', 'y_3338', 'lon', 'lat',
       'aksdb_othick_cum_best', 'x_pixel', 'y_pixel', 'band_1', 'band_2',
       'band_3', 'band_4', 'band_5', 'band_6', 'band_7', 'band_8', 'band_9',
       'band_10', 'band_11', 'band_12', 'band_13', 'band_14', 'band_15',
       'band_16', 'band_17', 'band_18', 'band_19', 'band_20', 'band_21',
       'band_22', 'band_23', 'band_24', 'band_25', 'band_26', 'band_27',
       'band_28', 'band_29', 'band_30', 'band_31', 'band_32', 'band_33',
       'band_34', 'band_35', 'band_36', 'band_37', 'band_38', 'band_39',
       'band_40', 'band_41', 'band_42', 'band_43', 'band_44', 'band_45',
       'band_46', 'band_47', 'band_48', 'band_49', 'band_50', 'band_51',
       'band_52', 'band_53', 'band_54', 'band_55', 'band_56', 'band_57',
       'band_58', 'band_59', 'band_60', 'aksdb_pf1m_bin', 'aksdb_pf1m_bin_gt',
       'geometry'],
      dtype='object')


In [8]:
# Append the tax_column to input data
# Visualize the head
gt_df = pd.DataFrame(gt_data)

# Then merge with your existing dataframe, keeping only necessary columns
point_data_gdf = point_data_gdf.merge(
    gt_df[['id', 'tax_order']], 
    on='id', 
    how='left'  # Use 'left', 'right', 'inner', or 'outer' depending on your needs
)

print(point_data_gdf.head(5))

   id  aksdb_dts         x_3338        y_3338         lon        lat  \
0   0 2003-07-16 -400760.487499  1.956308e+06 -163.285060  67.312360   
1   1 2002-07-14 -453698.480247  1.839239e+06 -164.084567  66.200267   
2   2 2017-08-26  -50769.743709  1.014299e+06 -154.889633  59.120601   
3   3 2012-08-18  515319.312000  1.754689e+06 -142.882002  65.362452   
4   4 2003-07-17 -471679.424115  1.780700e+06 -164.278140  65.655780   

  aksdb_othick_cum_best      x_pixel      y_pixel  band_1  ...  band_55  \
0                   nan  4923.451250  4369.707042     358  ...     2183   
1                  26.0  4629.651975  1076.614591     558  ...     1601   
2                  17.0  4922.525629  3570.578330     687  ...     2641   
3                  40.0  1531.431200  4531.604990     314  ...     2101   
4                   nan  2831.557588  1930.543496     493  ...     2511   

   band_56  band_57  band_58  band_59  band_60  aksdb_pf1m_bin  \
0     2323     2154     1178       18       51    

In [9]:
#Eliminate all the NA and multi-tax points
print('Shape of raw df: ', point_data_gdf.shape)
point_data_gdf = point_data_gdf[point_data_gdf['tax_order'].notna() & ~point_data_gdf['tax_order'].apply(lambda x: isinstance(x, list))]

# Convert 'tax_order' column to string
point_data_gdf['tax_order'] = point_data_gdf['tax_order'].astype('string')
print('Shape after eliminating NA and multi-tax points: ', point_data_gdf.shape)

Shape of raw df:  (37249, 73)
Shape after eliminating NA and multi-tax points:  (32143, 73)


In [10]:
# Eliminate alfisols, aridisols, oxisols
to_remove = ['Alfisols', 'Aridisols', 'Oxisols']
point_data_gdf = point_data_gdf[~point_data_gdf['tax_order'].isin(to_remove)]

print('Shape of gdf post elimination: ', point_data_gdf.shape)

Shape of gdf post elimination:  (32132, 73)


In [11]:
#Convert 'tax_order' column to an int
point_data_gdf['tax_order'] = point_data_gdf['tax_order'].str.strip()
unique_taxa = sorted(point_data_gdf['tax_order'].unique())
print('Unique Taxa: ', unique_taxa)
tax_order_dict = {name: idx for idx, name in enumerate(unique_taxa)}
idx_to_name_tax_dict = {idx: name for name, idx in tax_order_dict.items()}
point_data_gdf['tax_order_num'] = point_data_gdf['tax_order'].map(tax_order_dict)

Unique Taxa:  ['Andisols', 'Entisols', 'Gelisols', 'Histosols', 'Inceptisols', 'Mollisols', 'Spodosols']


In [12]:
# Define random forest run function
def run_rf(all_ids, kfold_indices, X, y, hyperparameters, save_preds=False, save_rf=False):
    indices = np.array(all_ids)
    
    fold_accuracies = []
    fold_precisions = []
    fold_recalls = []
    fold_f1s = []
    class_precisions = {}
    class_recalls = {}
    
    unique_classes = np.unique(y)
    
    for class_label in unique_classes:
        class_precisions[class_label] = []
        class_recalls[class_label] = []
    
    for fold, train_test_dict in kfold_indices.items():
        print('Training Fold: ', fold)

        train_indices = train_test_dict['train']
        test_indices = train_test_dict['test']

        train_rows = np.where(np.isin(indices, train_indices))[0]
        test_rows = np.where(np.isin(indices, test_indices))[0]

        X_train = X[train_rows]
        X_test = X[test_rows]
        y_train = y[train_rows]
        y_test = y[test_rows] 

        print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")
        print(f"y_train shape: {y_train.shape}, y_test shape: {y_test.shape}")

        est = hyperparameters['est']
        verbose = hyperparameters['verbose']
        n_jobs = hyperparameters['n_jobs']
        rf = RandomForestClassifier(n_estimators=est, verbose=verbose, n_jobs=n_jobs)
        rf.fit(X_train, y_train.ravel())
        y_pred = rf.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='macro')
        recall = recall_score(y_test, y_pred, average='macro')
        f1 = f1_score(y_test, y_pred, average='macro')
        precisions_per_class = precision_score(y_test, y_pred, average=None)
        recalls_per_class = recall_score(y_test, y_pred, average=None)
        
        fold_accuracies.append(accuracy)
        fold_precisions.append(precision)
        fold_recalls.append(recall)
        fold_f1s.append(f1)
        
        unique_y_test = np.unique(y_test)
        for i, class_label in enumerate(np.unique(y_test)):
            class_precision = precisions_per_class[i]
            class_recall = recalls_per_class[i]
            class_precisions[class_label].append(class_precision)
            class_recalls[class_label].append(class_recall)
        
        fold_num = int(fold.split('_')[-1])
        print(f"Fold {fold_num} Accuracy: {accuracy:.4f}")
        print(f"Fold {fold_num} Precision: {precision:.4f}")
        print(f"Fold {fold_num} Recall: {recall:.4f}")
        print(f"Fold {fold_num} F1: {f1:.4f}")
        
        if save_preds:
            test_ids = indices[np.isin(indices, test_indices)]
            preds = {
                "id": test_ids,  
                "gt": y_test.flatten(), 
                "pred": y_pred  
            }
            df_predictions = pd.DataFrame(preds)
            preds_outpath = f"test_predictions_{fold}.csv"
            df_predictions.to_csv(preds_outpath, index=False)

            print(f"Test predictions saved to {preds_outpath}")
            
        if save_rf:
            model_file = f"rf_weights_{fold}.pkl"
            with open(model_file, "wb") as f:
                pickle.dump(rf, f)
            print(f"Model saved to {model_file}")
        
    return (
        fold_accuracies, fold_precisions, fold_recalls, fold_f1s,
        class_precisions, class_recalls
    )

## Experiment Random Forest 1 (RF-1)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: R,G,B
    - Covariates: Lat, Lon
    - Order: B, G, R, Lon, Lat
- Output: 
    - Soil tax classification, multi-class classification
- Preprocessing: removal of low count classes, removal of NA


In [20]:
# Loading in data
channel_names = ['band_25', 'band_26', 'band_27']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['tax_order_num']

X = point_data_gdf[x_vars].to_numpy()
y = point_data_gdf[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

indices = point_data_gdf['id'].to_list()

X shape:  (32135, 5)
Y shape:  (32135, 1)


In [10]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [11]:
(
    accuracy, precision, recall, f1,
    class_precisions, class_recalls
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=True)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
}

for class_idx, precision_val_list in class_precisions.items():
    recall_val_list = class_recalls[class_idx]
    p_key = "Precision_" + idx_to_name_tax_dict[class_idx]
    r_key = "Recall_" + idx_to_name_tax_dict[class_idx]
    results[p_key] = precision_val_list
    results[r_key] = recall_val_list

results_df = pd.DataFrame(results)
result_outpath = "RF-1_tax-order_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

Training Fold:  fold_0
X_train shape: (25697, 5), X_test shape: (6438, 5)
y_train shape: (25697, 1), y_test shape: (6438, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.7s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    1.4s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


Fold 0 Accuracy: 0.5446
Fold 0 Precision: 0.4716
Fold 0 Recall: 0.4319
Fold 0 F1: 0.4386
Model saved to rf_weights_fold_0.pkl
Training Fold:  fold_1
X_train shape: (25901, 5), X_test shape: (6234, 5)
y_train shape: (25901, 1), y_test shape: (6234, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.6s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    1.4s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


Fold 1 Accuracy: 0.5417
Fold 1 Precision: 0.4956
Fold 1 Recall: 0.4403
Fold 1 F1: 0.4512
Model saved to rf_weights_fold_1.pkl
Training Fold:  fold_2
X_train shape: (26118, 5), X_test shape: (6017, 5)
y_train shape: (26118, 1), y_test shape: (6017, 1)


...

Fold 2 Accuracy: 0.5553
Fold 2 Precision: 0.4706
Fold 2 Recall: 0.4354
Fold 2 F1: 0.4417
Model saved to rf_weights_fold_2.pkl
Training Fold:  fold_3
X_train shape: (25309, 5), X_test shape: (6826, 5)
y_train shape: (25309, 1), y_test shape: (6826, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.6s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    1.3s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


Fold 3 Accuracy: 0.5762
Fold 3 Precision: 0.5270
Fold 3 Recall: 0.4593
Fold 3 F1: 0.4701
Model saved to rf_weights_fold_3.pkl
Training Fold:  fold_4
X_train shape: (25515, 5), X_test shape: (6620, 5)
y_train shape: (25515, 1), y_test shape: (6620, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.6s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    1.3s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


Fold 4 Accuracy: 0.5498
Fold 4 Precision: 0.5098
Fold 4 Recall: 0.4440
Fold 4 F1: 0.4569
Model saved to rf_weights_fold_4.pkl
Metrics saved to RF-1_tax-order_results.csv


## Experiment Random Forest 5 (RF-5)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: N/A
    - Covariates: Lat, Lon
    - Order: Lon, Lat
- Output: 
    - Soil tax classification, multi-class classification
- Preprocessing: removal of low count classes, removal of NA

In [14]:
# Loading in data
covariates = ['lon', 'lat']
x_vars = covariates
y_var = ['tax_order_num']

X = point_data_gdf[x_vars].to_numpy()
y = point_data_gdf[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

indices = point_data_gdf['id'].to_list()

X shape:  (32135, 2)
Y shape:  (32135, 1)


In [15]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [16]:
(
    accuracy, precision, recall, f1,
    class_precisions, class_recalls
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=True)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
}

for class_idx, precision_val_list in class_precisions.items():
    recall_val_list = class_recalls[class_idx]
    p_key = "Precision_" + idx_to_name_tax_dict[class_idx]
    r_key = "Recall_" + idx_to_name_tax_dict[class_idx]
    results[p_key] = precision_val_list
    results[r_key] = recall_val_list

results_df = pd.DataFrame(results)
result_outpath = "RF-5_tax-order_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

Training Fold:  fold_0
X_train shape: (25697, 2), X_test shape: (6438, 2)
y_train shape: (25697, 1), y_test shape: (6438, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.4s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.8s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


Fold 0 Accuracy: 0.4685
Fold 0 Precision: 0.3806
Fold 0 Recall: 0.3744
Fold 0 F1: 0.3733
Model saved to rf_weights_fold_0.pkl
Training Fold:  fold_1
X_train shape: (25901, 2), X_test shape: (6234, 2)
y_train shape: (25901, 1), y_test shape: (6234, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.4s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.9s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


Fold 1 Accuracy: 0.4827
Fold 1 Precision: 0.4171
Fold 1 Recall: 0.4027
Fold 1 F1: 0.4043
Model saved to rf_weights_fold_1.pkl
Training Fold:  fold_2
X_train shape: (26118, 2), X_test shape: (6017, 2)
y_train shape: (26118, 1), y_test shape: (6017, 1)


...

Fold 2 Accuracy: 0.4898
Fold 2 Precision: 0.3949
Fold 2 Recall: 0.3739
Fold 2 F1: 0.3784
Model saved to rf_weights_fold_2.pkl
Training Fold:  fold_3
X_train shape: (25309, 2), X_test shape: (6826, 2)
y_train shape: (25309, 1), y_test shape: (6826, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.4s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.9s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


Fold 3 Accuracy: 0.5223
Fold 3 Precision: 0.4451
Fold 3 Recall: 0.4156
Fold 3 F1: 0.4213
Model saved to rf_weights_fold_3.pkl
Training Fold:  fold_4
X_train shape: (25515, 2), X_test shape: (6620, 2)
y_train shape: (25515, 1), y_test shape: (6620, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.4s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.9s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


Fold 4 Accuracy: 0.4968
Fold 4 Precision: 0.4565
Fold 4 Recall: 0.4124
Fold 4 F1: 0.4244
Model saved to rf_weights_fold_4.pkl
Metrics saved to RF-5_tax-order_results.csv


## Experiment Random Forest 2 (RF-2)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Inputs: 
    - Channels: 25, 26, 27, 28, 29, 30, 31, 33, 34
    - Covariates: Lat, Lon
    - Intuition: Basically all the midsummer channels
    - Order: B, G, R, rededge1, rededge2, rededge3, nir, swir1, 
- Output: 
    - Soil tax classification, multi-class classification
- Preprocessing: removal of low count classes, removal of NA

In [9]:
# Loading in data 
channel_names = ['band_25', 'band_26', 'band_27', 'band_28', 'band_29', 'band_30', 'band_31', 'band_33', 'band_34']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['tax_order_num']

X = point_data_gdf[x_vars].to_numpy()
y = point_data_gdf[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

#load the order of the indices
indices = point_data_gdf['id'].to_list()

X shape:  (32135, 11)
Y shape:  (32135, 1)


In [10]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [12]:
(
    accuracy, precision, recall, f1,
    class_precisions, class_recalls
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=True)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
}

for class_idx, precision_val_list in class_precisions.items():
    recall_val_list = class_recalls[class_idx]
    p_key = "Precision_" + idx_to_name_tax_dict[class_idx]
    r_key = "Recall_" + idx_to_name_tax_dict[class_idx]
    results[p_key] = precision_val_list
    results[r_key] = recall_val_list

results_df = pd.DataFrame(results)
result_outpath = "results/SFold_taxOrderPred/RF-2_tax-order_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

Training Fold:  fold_0
X_train shape: (25697, 11), X_test shape: (6438, 11)
y_train shape: (25697, 1), y_test shape: (6438, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.8s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    1.9s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


Fold 0 Accuracy: 0.5710
Fold 0 Precision: 0.5919
Fold 0 Recall: 0.4423
Fold 0 F1: 0.4595
Model saved to rf_weights_fold_0.pkl
Training Fold:  fold_1
X_train shape: (25901, 11), X_test shape: (6234, 11)
y_train shape: (25901, 1), y_test shape: (6234, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.9s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    1.9s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


Fold 1 Accuracy: 0.5642
Fold 1 Precision: 0.5243
Fold 1 Recall: 0.4581
Fold 1 F1: 0.4690
Model saved to rf_weights_fold_1.pkl
Training Fold:  fold_2
X_train shape: (26118, 11), X_test shape: (6017, 11)
y_train shape: (26118, 1), y_test shape: (6017, 1)


...

Fold 2 Accuracy: 0.5715
Fold 2 Precision: 0.4901
Fold 2 Recall: 0.4459
Fold 2 F1: 0.4544
Model saved to rf_weights_fold_2.pkl
Training Fold:  fold_3
X_train shape: (25309, 11), X_test shape: (6826, 11)
y_train shape: (25309, 1), y_test shape: (6826, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.8s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    1.8s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


Fold 3 Accuracy: 0.5876
Fold 3 Precision: 0.5594
Fold 3 Recall: 0.4709
Fold 3 F1: 0.4803
Model saved to rf_weights_fold_3.pkl
Training Fold:  fold_4
X_train shape: (25515, 11), X_test shape: (6620, 11)
y_train shape: (25515, 1), y_test shape: (6620, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.8s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    2.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


Fold 4 Accuracy: 0.5810
Fold 4 Precision: 0.5716
Fold 4 Recall: 0.4638
Fold 4 F1: 0.4781
Model saved to rf_weights_fold_4.pkl
Metrics saved to results/SFold_taxOrderPred/RF-2_tax-order_results.csv


## Experiment Random Forest 7 (RF-7)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: R,G,B, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, Lat, Lon
    - Covariates: Lat, Lon
    - Order: B, G, R, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, Lon, Lat
- Output: 
    - Soil tax classification, multi-class classification
- Preprocessing: removal of low count classes, removal of NA


In [31]:
# Loading in covars
covar_pt = '../data/point_pixel_ifsar_derivs_v1.csv'
covar_gdf = gpd.read_file(covar_pt)
print(covar_gdf.columns)
covar_gdf = covar_gdf[['id', 'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']]

Index(['field_1', 'id', 'aksdb_dts', 'x_3338', 'y_3338', 'lon', 'lat',
       'aksdb_othick_cum_best', 'tax_order', 'x_pixel', 'y_pixel', 'grid_id',
       'aksdb_pf1m_bin', 'aspct_4_band_1', 'maxc_4_band_1', 'spi_band_1',
       'tpi_4_band_1', 'elevation_full_10m_3338_band_1', 'sl_4_band_1',
       'swi_10_band_1', 'geometry'],
      dtype='object')


In [36]:
print(point_data_gdf.shape)
print(covar_gdf.shape)

(32135, 74)
(37252, 8)


In [39]:
point_data_gdf['id'] = point_data_gdf['id'].astype('int64')
covar_gdf['id'] = covar_gdf['id'].astype('int64')
merged_df = pd.merge(point_data_gdf, covar_gdf, on='id', how='inner')

In [40]:
# Loading in data
channel_names = ['band_25', 'band_26', 'band_27', 'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['tax_order_num']

X = merged_df[x_vars].to_numpy()
y = merged_df[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

indices = merged_df['id'].to_list()

X shape:  (32135, 12)
Y shape:  (32135, 1)


In [41]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [43]:
(
    accuracy, precision, recall, f1,
    class_precisions, class_recalls
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
}

for class_idx, precision_val_list in class_precisions.items():
    recall_val_list = class_recalls[class_idx]
    p_key = "Precision_" + idx_to_name_tax_dict[class_idx]
    r_key = "Recall_" + idx_to_name_tax_dict[class_idx]
    results[p_key] = precision_val_list
    results[r_key] = recall_val_list

results_df = pd.DataFrame(results)
result_outpath = "results/SFold_taxOrderPred/RF-7_tax-order_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

Training Fold:  fold_0
X_train shape: (25697, 12), X_test shape: (6438, 12)
y_train shape: (25697, 1), y_test shape: (6438, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    4.2s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    8.7s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


Fold 0 Accuracy: 0.5974
Fold 0 Precision: 0.5670
Fold 0 Recall: 0.4607
Fold 0 F1: 0.4721
Training Fold:  fold_1
X_train shape: (25901, 12), X_test shape: (6234, 12)
y_train shape: (25901, 1), y_test shape: (6234, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    3.8s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    6.6s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


Fold 1 Accuracy: 0.5927
Fold 1 Precision: 0.5408
Fold 1 Recall: 0.4851
Fold 1 F1: 0.4935
Training Fold:  fold_2
X_train shape: (26118, 12), X_test shape: (6017, 12)
y_train shape: (26118, 1), y_test shape: (6017, 1)


...

Fold 2 Accuracy: 0.5963
Fold 2 Precision: 0.5001
Fold 2 Recall: 0.4648
Fold 2 F1: 0.4734
Training Fold:  fold_3
X_train shape: (25309, 12), X_test shape: (6826, 12)
y_train shape: (25309, 1), y_test shape: (6826, 1)


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    1.9s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    4.4s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.2s finished


Fold 3 Accuracy: 0.6160
Fold 3 Precision: 0.6191
Fold 3 Recall: 0.4984
Fold 3 F1: 0.5138
Training Fold:  fold_4
X_train shape: (25515, 12), X_test shape: (6620, 12)
y_train shape: (25515, 1), y_test shape: (6620, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    4.6s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


Fold 4 Accuracy: 0.6056
Fold 4 Precision: 0.6288
Fold 4 Recall: 0.4938
Fold 4 F1: 0.5217
Metrics saved to results/SFold_taxOrderPred/RF-7_tax-order_results.csv


## Experiment Random Forest 8 (RF-8)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: 25, 26, 27, 28, 29, 30, 31, 33, 34, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, Lat, Lon
    - Covariates: Lat, Lon
    - Order: B, G, R, ededge1, rededge2, rededge3, nir, swir1, ppt_annual, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, Lon, Lat
- Output: 
    - Soil tax classification, multi-class classification
- Preprocessing: removal of low count classes, removal of NA


In [9]:
# Loading in covars
covar_pt = '../data/point_pixel_ifsar_derivs_v1.csv'
covar_gdf = gpd.read_file(covar_pt)
print(covar_gdf.columns)
covar_gdf = covar_gdf[['id', 'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']]

Index(['field_1', 'id', 'aksdb_dts', 'x_3338', 'y_3338', 'lon', 'lat',
       'aksdb_othick_cum_best', 'tax_order', 'x_pixel', 'y_pixel', 'grid_id',
       'aksdb_pf1m_bin', 'aspct_4_band_1', 'maxc_4_band_1', 'spi_band_1',
       'tpi_4_band_1', 'elevation_full_10m_3338_band_1', 'sl_4_band_1',
       'swi_10_band_1'],
      dtype='object')


In [10]:
point_data_gdf['id'] = point_data_gdf['id'].astype('int64')
covar_gdf['id'] = covar_gdf['id'].astype('int64')
merged_df = pd.merge(point_data_gdf, covar_gdf, on='id', how='inner')

In [11]:
# Loading in data
channel_names = ['band_25', 'band_26', 'band_27', 'band_28', 'band_29', 'band_30', 'band_31', 'band_33', 'band_34', 'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['tax_order_num']

X = merged_df[x_vars].to_numpy()
y = merged_df[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

indices = merged_df['id'].to_list()

X shape:  (32132, 18)
Y shape:  (32132, 1)


In [12]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [13]:
(
    accuracy, precision, recall, f1,
    class_precisions, class_recalls
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=True)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
}

for class_idx, precision_val_list in class_precisions.items():
    recall_val_list = class_recalls[class_idx]
    p_key = "Precision_" + idx_to_name_tax_dict[class_idx]
    r_key = "Recall_" + idx_to_name_tax_dict[class_idx]
    results[p_key] = precision_val_list
    results[r_key] = recall_val_list

results_df = pd.DataFrame(results)
result_outpath = "results/SFold_taxOrderPred/RF-8_tax-order_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

Training Fold:  fold_0
X_train shape: (25694, 18), X_test shape: (6438, 18)
y_train shape: (25694, 1), y_test shape: (6438, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    3.3s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    6.7s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


Fold 0 Accuracy: 0.6066
Fold 0 Precision: 0.6161
Fold 0 Recall: 0.4684
Fold 0 F1: 0.4846
Model saved to rf_weights_fold_0.pkl
Training Fold:  fold_1
X_train shape: (25900, 18), X_test shape: (6232, 18)
y_train shape: (25900, 1), y_test shape: (6232, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.7s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    6.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


Fold 1 Accuracy: 0.6093
Fold 1 Precision: 0.5589
Fold 1 Recall: 0.4904
Fold 1 F1: 0.5009
Model saved to rf_weights_fold_1.pkl
Training Fold:  fold_2
X_train shape: (26115, 18), X_test shape: (6017, 18)
y_train shape: (26115, 1), y_test shape: (6017, 1)


...

Fold 2 Accuracy: 0.6058
Fold 2 Precision: 0.5123
Fold 2 Recall: 0.4734
Fold 2 F1: 0.4833
Model saved to rf_weights_fold_2.pkl
Training Fold:  fold_3
X_train shape: (25307, 18), X_test shape: (6825, 18)
y_train shape: (25307, 1), y_test shape: (6825, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.6s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    5.9s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


Fold 3 Accuracy: 0.6232
Fold 3 Precision: 0.6440
Fold 3 Recall: 0.5076
Fold 3 F1: 0.5223
Model saved to rf_weights_fold_3.pkl
Training Fold:  fold_4
X_train shape: (25512, 18), X_test shape: (6620, 18)
y_train shape: (25512, 1), y_test shape: (6620, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.8s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    6.2s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


Fold 4 Accuracy: 0.6121
Fold 4 Precision: 0.6216
Fold 4 Recall: 0.4862
Fold 4 F1: 0.5041
Model saved to rf_weights_fold_4.pkl
Metrics saved to results/SFold_taxOrderPred/RF-8_tax-order_results.csv


## Experiment Random Forest 9 (RF-9)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: R,G,B, ppt_annual, tmean_swi, tmin_january
    - Covariates: Lat, Lon
    - Order: B, G, R, ppt_annual, tmean_swi, tmin_january, Lon, Lat
- Output: 
    - Soil tax classification, multi-class classification
- Preprocessing: removal of low count classes, removal of NA


In [11]:
covar_pt = '../data/point_pixel_climate_v1.csv'
covar_gdf = pd.read_csv(covar_pt)
print(covar_gdf.columns)
covar_gdf = covar_gdf[['id', 'ppt_annual_band_1', 'tmean_swi_band_1', 'tmin_january_band_1']]

Index(['Unnamed: 0', 'id', 'aksdb_dts', 'x_3338', 'y_3338', 'lon', 'lat',
       'aksdb_othick_cum_best', 'tax_order', 'x_pixel', 'y_pixel', 'grid_id',
       'aksdb_pf1m_bin', 'ppt_annual_band_1', 'tmean_swi_band_1',
       'tmin_january_band_1'],
      dtype='object')


In [12]:
merged_df = point_data_gdf.merge(covar_gdf, on='id', how='inner')

In [21]:
# Loading in data
channel_names = ['band_25', 'band_26', 'band_27', 'ppt_annual_band_1','tmean_swi_band_1','tmin_january_band_1']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['tax_order_num']

X = merged_df[x_vars].to_numpy()
y = merged_df[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

indices = merged_df['id'].to_list()

X shape:  (32135, 8)
Y shape:  (32135, 1)


In [22]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [24]:
(
    accuracy, precision, recall, f1,
    class_precisions, class_recalls
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
}

for class_idx, precision_val_list in class_precisions.items():
    recall_val_list = class_recalls[class_idx]
    p_key = "Precision_" + idx_to_name_tax_dict[class_idx]
    r_key = "Recall_" + idx_to_name_tax_dict[class_idx]
    results[p_key] = precision_val_list
    results[r_key] = recall_val_list

results_df = pd.DataFrame(results)
result_outpath = "results/SFold_taxOrderPred/RF-9_tax-order_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

Training Fold:  fold_0
X_train shape: (25697, 8), X_test shape: (6438, 8)
y_train shape: (25697, 1), y_test shape: (6438, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    1.7s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    3.6s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 0 Accuracy: 0.5687
Fold 0 Precision: 0.5154
Fold 0 Recall: 0.4475
Fold 0 F1: 0.4586
Training Fold:  fold_1
X_train shape: (25901, 8), X_test shape: (6234, 8)
y_train shape: (25901, 1), y_test shape: (6234, 1)


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.8s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    4.7s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 1 Accuracy: 0.5552
Fold 1 Precision: 0.5261
Fold 1 Recall: 0.4557
Fold 1 F1: 0.4641
Training Fold:  fold_2
X_train shape: (26118, 8), X_test shape: (6017, 8)
y_train shape: (26118, 1), y_test shape: (6017, 1)


...

[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    1.4s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    3.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 3 Accuracy: 0.5828
Fold 3 Precision: 0.5444
Fold 3 Recall: 0.4614
Fold 3 F1: 0.4721
Training Fold:  fold_4
X_train shape: (25515, 8), X_test shape: (6620, 8)
y_train shape: (25515, 1), y_test shape: (6620, 1)


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    1.3s


Fold 4 Accuracy: 0.5674
Fold 4 Precision: 0.5485
Fold 4 Recall: 0.4642
Fold 4 F1: 0.4863
Metrics saved to results/SFold_taxOrderPred/RF-9_tax-order_results.csv


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    2.9s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


## Experiment Random Forest 10 (RF-10)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: 25, 26, 27, 28, 29, 30, 31, 33, 34, ppt_annual, tmean_swi, tmin_january
    - Covariates: Lat, Lon
    - Order: B, G, R, rededge1, rededge2, rededge3, nir, swir1, ppt_annual, tmean_swi, tmin_january, Lon, Lat
- Output: 
    - Soil tax classification, multi-class classification
- Preprocessing: removal of low count classes, removal of NA


In [26]:
covar_pt = '../data/point_pixel_climate_v1.csv'
covar_gdf = pd.read_csv(covar_pt)
print(covar_gdf.columns)
covar_gdf = covar_gdf[['id', 'ppt_annual_band_1', 'tmean_swi_band_1', 'tmin_january_band_1']]

Index(['Unnamed: 0', 'id', 'aksdb_dts', 'x_3338', 'y_3338', 'lon', 'lat',
       'aksdb_othick_cum_best', 'tax_order', 'x_pixel', 'y_pixel', 'grid_id',
       'aksdb_pf1m_bin', 'ppt_annual_band_1', 'tmean_swi_band_1',
       'tmin_january_band_1'],
      dtype='object')


In [27]:
merged_df = point_data_gdf.merge(covar_gdf, on='id', how='inner')

In [28]:
# Loading in data
channel_names = ['band_25', 'band_26', 'band_27', 'band_28', 'band_29', 'band_30', 'band_31', 'band_33', 'band_34', 'ppt_annual_band_1','tmean_swi_band_1','tmin_january_band_1']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['tax_order_num']

X = merged_df[x_vars].to_numpy()
y = merged_df[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

indices = merged_df['id'].to_list()

X shape:  (32135, 14)
Y shape:  (32135, 1)


In [29]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [30]:
(
    accuracy, precision, recall, f1,
    class_precisions, class_recalls
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
}

for class_idx, precision_val_list in class_precisions.items():
    recall_val_list = class_recalls[class_idx]
    p_key = "Precision_" + idx_to_name_tax_dict[class_idx]
    r_key = "Recall_" + idx_to_name_tax_dict[class_idx]
    results[p_key] = precision_val_list
    results[r_key] = recall_val_list

results_df = pd.DataFrame(results)
result_outpath = "results/SFold_taxOrderPred/RF-10_tax-order_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

Training Fold:  fold_0
X_train shape: (25697, 14), X_test shape: (6438, 14)
y_train shape: (25697, 1), y_test shape: (6438, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.4s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    5.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 0 Accuracy: 0.5943
Fold 0 Precision: 0.5628
Fold 0 Recall: 0.4557
Fold 0 F1: 0.4692
Training Fold:  fold_1
X_train shape: (25901, 14), X_test shape: (6234, 14)
y_train shape: (25901, 1), y_test shape: (6234, 1)


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    3.2s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    5.8s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 1 Accuracy: 0.5792
Fold 1 Precision: 0.5094
Fold 1 Recall: 0.4621
Fold 1 F1: 0.4686
Training Fold:  fold_2
X_train shape: (26118, 14), X_test shape: (6017, 14)
y_train shape: (26118, 1), y_test shape: (6017, 1)


...

[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    1.8s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    4.3s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.2s finished


Fold 3 Accuracy: 0.6034
Fold 3 Precision: 0.5871
Fold 3 Recall: 0.4869
Fold 3 F1: 0.4968
Training Fold:  fold_4
X_train shape: (25515, 14), X_test shape: (6620, 14)
y_train shape: (25515, 1), y_test shape: (6620, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.0s


Fold 4 Accuracy: 0.5917
Fold 4 Precision: 0.5726
Fold 4 Recall: 0.4705
Fold 4 F1: 0.4856
Metrics saved to results/SFold_taxOrderPred/RF-10_tax-order_results.csv


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    4.5s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


## Experiment Random Forest 11 (RF-11)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: 25, 26, 27, 28, 29, 30, 31, 33, 34, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, ppt_annual, tmean_swi, tmin_january
    - Covariates: Lat, Lon
    - Order: basically satellite channels, env covars like DEM, climate covars
        - B, G, R, rededge1, rededge2, rededge3, nir, swir1, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, ppt_annual, tmean_swi, tmin_january, Lon, Lat
- Output: 
    - Soil tax classification, multi-class classification
- Preprocessing: removal of low count classes, removal of NA


In [9]:
# Loading in covars
topo_covar_pt = '../data/point_pixel_ifsar_derivs_v1.csv'
topo_covar_gdf = gpd.read_file(topo_covar_pt)
topo_covar_gdf = topo_covar_gdf[['id', 'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']]
topo_covar_gdf['id'] = topo_covar_gdf['id'].astype('int64')

climate_covar_pt = '../data/point_pixel_climate_v1.csv'
climate_covar_gdf = pd.read_csv(climate_covar_pt)
climate_covar_gdf = climate_covar_gdf[['id', 'ppt_annual_band_1', 'tmean_swi_band_1', 'tmin_january_band_1']]

In [10]:
merged_df = point_data_gdf.merge(topo_covar_gdf, on='id', how='inner')
merged_df = merged_df.merge(climate_covar_gdf, on='id', how='inner')

In [11]:
# Loading in data
channel_names = ['band_25', 'band_26', 'band_27', 'band_28', 'band_29', 'band_30', 'band_31', 'band_33', 'band_34', 
                    'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 
                     'swi_10_band_1', 'tpi_4_band_1', 'ppt_annual_band_1', 'tmean_swi_band_1', 'tmin_january_band_1']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['tax_order_num']

X = merged_df[x_vars].to_numpy()
y = merged_df[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

indices = merged_df['id'].to_list()

X shape:  (32132, 21)
Y shape:  (32132, 1)


In [14]:
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [15]:
(
    accuracy, precision, recall, f1,
    class_precisions, class_recalls
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = True, save_rf=True)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
}

for class_idx, precision_val_list in class_precisions.items():
    recall_val_list = class_recalls[class_idx]
    p_key = "Precision_" + idx_to_name_tax_dict[class_idx]
    r_key = "Recall_" + idx_to_name_tax_dict[class_idx]
    results[p_key] = precision_val_list
    results[r_key] = recall_val_list

results_df = pd.DataFrame(results)
result_outpath = "results/SFold_taxOrderPred/RF-11_tax-order_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

Training Fold:  fold_0
X_train shape: (25694, 21), X_test shape: (6438, 21)
y_train shape: (25694, 1), y_test shape: (6438, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    3.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    6.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


Fold 0 Accuracy: 0.6126
Fold 0 Precision: 0.5623
Fold 0 Recall: 0.4726
Fold 0 F1: 0.4864
Test predictions saved to test_predictions_fold_0.csv
Model saved to rf_weights_fold_0.pkl
Training Fold:  fold_1
X_train shape: (25900, 21), X_test shape: (6232, 21)
y_train shape: (25900, 1), y_test shape: (6232, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.6s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    5.8s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


Fold 1 Accuracy: 0.6080
Fold 1 Precision: 0.5546
Fold 1 Recall: 0.4930
Fold 1 F1: 0.5028
Test predictions saved to test_predictions_fold_1.csv
Model saved to rf_weights_fold_1.pkl
Training Fold:  fold_2
X_train shape: (26115, 21), X_test shape: (6017, 21)
y_train shape: (26115, 1), y_test shape: (6017, 1)


...

Fold 2 Accuracy: 0.6129
Fold 2 Precision: 0.5141
Fold 2 Recall: 0.4791
Fold 2 F1: 0.4870
Test predictions saved to test_predictions_fold_2.csv
Model saved to rf_weights_fold_2.pkl
Training Fold:  fold_3
X_train shape: (25307, 21), X_test shape: (6825, 21)
y_train shape: (25307, 1), y_test shape: (6825, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.6s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    5.7s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


Fold 3 Accuracy: 0.6262
Fold 3 Precision: 0.6389
Fold 3 Recall: 0.5058
Fold 3 F1: 0.5184
Test predictions saved to test_predictions_fold_3.csv
Model saved to rf_weights_fold_3.pkl
Training Fold:  fold_4
X_train shape: (25512, 21), X_test shape: (6620, 21)
y_train shape: (25512, 1), y_test shape: (6620, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.5s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    5.6s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


Fold 4 Accuracy: 0.6168
Fold 4 Precision: 0.6472
Fold 4 Recall: 0.4947
Fold 4 F1: 0.5206
Test predictions saved to test_predictions_fold_4.csv
Model saved to rf_weights_fold_4.pkl
Metrics saved to results/SFold_taxOrderPred/RF-11_tax-order_results.csv


## Experiment Random Forest 12 (RF-12)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4
    - Covariates: Lat, Lon
    - Order: env covars like DEM ONLY
        - aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, Lon, Lat
- Output: 
    - Soil tax classification, multi-class classification
- Preprocessing: removal of low count classes, removal of NA


In [6]:
# Loading in covars
covar_pt = '/home/yaoyi/chen7924/peatland_permafrost/data/point_pixel_ifsar_derivs_v1.csv'
covar_gdf = gpd.read_file(covar_pt)
print(covar_gdf.columns)
covar_gdf = covar_gdf[['id', 'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']]

Index(['field_1', 'id', 'aksdb_dts', 'x_3338', 'y_3338', 'lon', 'lat',
       'aksdb_othick_cum_best', 'tax_order', 'x_pixel', 'y_pixel', 'grid_id',
       'aksdb_pf1m_bin', 'aspct_4_band_1', 'maxc_4_band_1', 'spi_band_1',
       'tpi_4_band_1', 'elevation_full_10m_3338_band_1', 'sl_4_band_1',
       'swi_10_band_1'],
      dtype='object')


In [13]:
point_data_gdf['id'] = point_data_gdf['id'].astype('int64')
covar_gdf['id'] = covar_gdf['id'].astype('int64')
merged_df = pd.merge(point_data_gdf, covar_gdf, on='id', how='inner')

In [14]:
channel_names = ['aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['tax_order_num']

X = merged_df[x_vars].to_numpy()
y = merged_df[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

indices = merged_df['id'].to_list()

X shape:  (32132, 9)
Y shape:  (32132, 1)


In [15]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [16]:
(
    accuracy, precision, recall, f1,
    class_precisions, class_recalls
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
}

for class_idx, precision_val_list in class_precisions.items():
    recall_val_list = class_recalls[class_idx]
    p_key = "Precision_" + idx_to_name_tax_dict[class_idx]
    r_key = "Recall_" + idx_to_name_tax_dict[class_idx]
    results[p_key] = precision_val_list
    results[r_key] = recall_val_list

results_df = pd.DataFrame(results)
result_outpath = "results/SFold_taxOrderPred/RF-12_tax-order_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

Training Fold:  fold_0
X_train shape: (25694, 9), X_test shape: (6438, 9)
y_train shape: (25694, 1), y_test shape: (6438, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.7s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    5.7s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.


Fold 0 Accuracy: 0.5788
Fold 0 Precision: 0.5204
Fold 0 Recall: 0.4522
Fold 0 F1: 0.4637
Training Fold:  fold_1
X_train shape: (25900, 9), X_test shape: (6232, 9)
y_train shape: (25900, 1), y_test shape: (6232, 1)


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.2s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    5.2s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


Fold 1 Accuracy: 0.5722
Fold 1 Precision: 0.5239
Fold 1 Recall: 0.4706
Fold 1 F1: 0.4764
Training Fold:  fold_2
X_train shape: (26115, 9), X_test shape: (6017, 9)
y_train shape: (26115, 1), y_test shape: (6017, 1)


...

Fold 2 Accuracy: 0.5828
Fold 2 Precision: 0.5052
Fold 2 Recall: 0.4594
Fold 2 F1: 0.4667
Training Fold:  fold_3
X_train shape: (25307, 9), X_test shape: (6825, 9)
y_train shape: (25307, 1), y_test shape: (6825, 1)


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.2s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    4.9s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


Fold 3 Accuracy: 0.6000
Fold 3 Precision: 0.5952
Fold 3 Recall: 0.4800
Fold 3 F1: 0.4958
Training Fold:  fold_4
X_train shape: (25512, 9), X_test shape: (6620, 9)
y_train shape: (25512, 1), y_test shape: (6620, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.2s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    5.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


Fold 4 Accuracy: 0.5863
Fold 4 Precision: 0.5684
Fold 4 Recall: 0.4791
Fold 4 F1: 0.5029
Metrics saved to results/SFold_taxOrderPred/RF-12_tax-order_results.csv


## Experiment Random Forest 13 (RF-13)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4
    - Order: env covars like DEM ONLY
        - aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4
- Output: 
    - Soil tax classification, multi-class classification
- Preprocessing: removal of low count classes, removal of NA


In [19]:
# Loading in covars
covar_pt = '/home/yaoyi/chen7924/peatland_permafrost/data/point_pixel_ifsar_derivs_v1.csv'
covar_gdf = gpd.read_file(covar_pt)
print(covar_gdf.columns)
covar_gdf = covar_gdf[['id', 'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']]

Index(['field_1', 'id', 'aksdb_dts', 'x_3338', 'y_3338', 'lon', 'lat',
       'aksdb_othick_cum_best', 'tax_order', 'x_pixel', 'y_pixel', 'grid_id',
       'aksdb_pf1m_bin', 'aspct_4_band_1', 'maxc_4_band_1', 'spi_band_1',
       'tpi_4_band_1', 'elevation_full_10m_3338_band_1', 'sl_4_band_1',
       'swi_10_band_1'],
      dtype='object')


In [20]:
point_data_gdf['id'] = point_data_gdf['id'].astype('int64')
covar_gdf['id'] = covar_gdf['id'].astype('int64')
merged_df = pd.merge(point_data_gdf, covar_gdf, on='id', how='inner')

In [21]:
channel_names = ['aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']
x_vars = channel_names 
y_var = ['tax_order_num']

X = merged_df[x_vars].to_numpy()
y = merged_df[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

indices = merged_df['id'].to_list()

X shape:  (32132, 7)
Y shape:  (32132, 1)


In [22]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [23]:
(
    accuracy, precision, recall, f1,
    class_precisions, class_recalls
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
}

for class_idx, precision_val_list in class_precisions.items():
    recall_val_list = class_recalls[class_idx]
    p_key = "Precision_" + idx_to_name_tax_dict[class_idx]
    r_key = "Recall_" + idx_to_name_tax_dict[class_idx]
    results[p_key] = precision_val_list
    results[r_key] = recall_val_list

results_df = pd.DataFrame(results)
result_outpath = "results/SFold_taxOrderPred/RF-13_tax-order_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

Training Fold:  fold_0
X_train shape: (25694, 7), X_test shape: (6438, 7)
y_train shape: (25694, 1), y_test shape: (6438, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    4.3s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.2s finished
/users/0/chen7924/.conda/envs/gp2/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/users/0/chen7924/.conda/envs/gp2/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` paramet

Fold 0 Accuracy: 0.4161
Fold 0 Precision: 0.2769
Fold 0 Recall: 0.2681
Fold 0 F1: 0.2629
Training Fold:  fold_1
X_train shape: (25900, 7), X_test shape: (6232, 7)
y_train shape: (25900, 1), y_test shape: (6232, 1)


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    1.8s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    3.9s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.2s finished


Fold 1 Accuracy: 0.4090
Fold 1 Precision: 0.3669
Fold 1 Recall: 0.2832
Fold 1 F1: 0.2767
Training Fold:  fold_2
X_train shape: (26115, 7), X_test shape: (6017, 7)
y_train shape: (26115, 1), y_test shape: (6017, 1)


...

Fold 2 Accuracy: 0.3972
Fold 2 Precision: 0.2669
Fold 2 Recall: 0.2640
Fold 2 F1: 0.2559
Training Fold:  fold_3
X_train shape: (25307, 7), X_test shape: (6825, 7)
y_train shape: (25307, 1), y_test shape: (6825, 1)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    1.8s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    4.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.2s finished
/users/0/chen7924/.conda/envs/gp2/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/users/0/chen7924/.conda/envs/gp2/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` paramet

Fold 3 Accuracy: 0.3764
Fold 3 Precision: 0.2674
Fold 3 Recall: 0.2588
Fold 3 F1: 0.2463
Training Fold:  fold_4
X_train shape: (25512, 7), X_test shape: (6620, 7)
y_train shape: (25512, 1), y_test shape: (6620, 1)


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    4.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished


Fold 4 Accuracy: 0.4166
Fold 4 Precision: 0.2990
Fold 4 Recall: 0.2681
Fold 4 F1: 0.2628
Metrics saved to results/SFold_taxOrderPred/RF-13_tax-order_results.csv
